In [9]:
import torch
import torch.nn as nn

from src.gtsrb import NUM_CLASSES, get_dataloaders, save_predictions

In [10]:
train_loader, val_loader, test_loader = get_dataloaders(img_size=32, batch_size=128)

In [11]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([128, 3, 32, 32])
torch.Size([128])


In [12]:
model = nn.Sequential(

    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Flatten(),

    nn.Linear(64 * 8 * 8, 128),
    nn.ReLU(),

    nn.Linear(128, NUM_CLASSES)
)

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=4096, out_features=128, bias=True)
  (8): ReLU()
  (9): Linear(in_features=128, out_features=43, bias=True)
)

In [14]:
criterion = nn.CrossEntropyLoss()

In [15]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [16]:
epochs = 5

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")


    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()

            total += labels.size(0)

    accuracy = 100 * correct / total

    print(f"Validation Accuracy: {accuracy:.2f}%")

Epoch 1, Loss: 287.0742
Validation Accuracy: 80.76%
Epoch 2, Loss: 60.9693
Validation Accuracy: 93.86%
Epoch 3, Loss: 26.5821
Validation Accuracy: 95.18%
Epoch 4, Loss: 15.0758
Validation Accuracy: 97.22%
Epoch 5, Loss: 10.0400
Validation Accuracy: 97.50%


In [17]:
model.eval()

all_preds = []

with torch.no_grad():

    for images, _ in test_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        all_preds.append(predictions.cpu())

y_pred = torch.cat(all_preds)

In [18]:
save_predictions(
    y_pred,
    "results/predicoes_baseline.csv",
    experiment_name="CNN Baseline"
)